# Summary: D1 and basic services

This notebook compares global income deciles by the share of each decile living in countries with low national coverage of electricity, safe drinking water, or internet use.

A country is classified as low coverage when the indicator is below 50%. This measures **where people live**; it does not identify which households within a country have the service. The three indicators use their latest broadly covered years, so the years differ.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'indicator_analysis_helpers.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
from indicator_analysis_helpers import D1_FILE, fetch_indicator

D1_PATH = D1_FILE if D1_FILE.exists() else NOTEBOOK_DIR.parent / D1_FILE.name
distribution = pd.read_csv(D1_PATH)
for column in ('year', 'welf', 'pop'):
    distribution[column] = pd.to_numeric(distribution[column], errors='coerce')
distribution = distribution.dropna(subset=['year', 'welf', 'pop'])

indicator_sources = {
    'Electricity': {'dataset': 'WB_WDI', 'indicator': 'WB_WDI_EG_ELC_ACCS_ZS', 'year': 2023},
    'Safe drinking water': {'dataset': 'FAO_AS', 'indicator': 'FAO_AS_4114', 'year': 2023},
    'Internet use': {'dataset': 'ITU_DH', 'indicator': 'IT_NET_USER', 'year': 2024},
}
LOW_COVERAGE_THRESHOLD = 50
result_rows = []
coverage_rows = []

for service, source in indicator_sources.items():
    year = source['year']
    global_distribution = distribution[distribution['year'].eq(year)].sort_values('welf', kind='stable').copy()
    global_population = global_distribution['pop'].sum()
    global_distribution['decile'] = np.minimum(
        np.ceil(global_distribution['pop'].cumsum() / (global_population / 10)),
        10,
    ).astype(int)

    country_indicator = fetch_indicator(source['dataset'], source['indicator'])
    country_indicator = country_indicator[country_indicator['year'].eq(year)].drop_duplicates('code')
    country_indicator = country_indicator.set_index('code')['indicator_value']
    global_distribution = global_distribution.merge(
        country_indicator.rename('national_coverage'),
        left_on='code',
        right_index=True,
        how='left',
    )

    for decile, decile_group in global_distribution.groupby('decile'):
        observed = decile_group.dropna(subset=['national_coverage'])
        covered_population = observed['pop'].sum()
        low_coverage_population = observed.loc[
            observed['national_coverage'].lt(LOW_COVERAGE_THRESHOLD), 'pop'
        ].sum()
        result_rows.append({
            'Decile': f'D{decile}',
            'Service': service,
            'Year': year,
            'Share living in countries below 50% (%)': 100 * low_coverage_population / covered_population,
        })
        coverage_rows.append({
            'Decile': f'D{decile}',
            'Service': service,
            'Year': year,
            'Population with country data (%)': 100 * covered_population / decile_group['pop'].sum(),
        })

result_table = pd.DataFrame(result_rows).pivot(
    index='Decile', columns=['Service', 'Year'], values='Share living in countries below 50% (%)'
).reindex([f'D{number}' for number in range(1, 11)])
coverage_table = pd.DataFrame(coverage_rows).pivot(
    index='Decile', columns=['Service', 'Year'], values='Population with country data (%)'
).reindex([f'D{number}' for number in range(1, 11)])

print('Share of each global income decile living in countries with national service coverage below 50%')
display(result_table.round(2))
print('Population coverage of the indicator data, by decile')
display(coverage_table.round(1))

Share of each global income decile living in countries with national service coverage below 50%


Service,Electricity,Safe drinking water,Internet use
Year,2023,2023,2024
Decile,,,
D1,30.57,3.90,74.00
D2,7.67,1.47,33.42
D3,3.42,0.81,16.40
D4,2.24,0.64,11.82
D5,1.54,0.56,8.80
D6,1.09,0.44,6.20
D7,0.62,0.30,3.54
D8,0.31,0.15,1.47


Population coverage of the indicator data, by decile


Service,Electricity,Safe drinking water,Internet use
Year,2023,2023,2024
Decile,,,
D1,100.0,100.0,90.6
D2,100.0,100.0,94.1
D3,100.0,100.0,96.2
D4,100.0,100.0,97.2
D5,100.0,100.0,97.6
D6,100.0,99.9,98.1
D7,99.9,99.8,98.8
D8,99.6,99.4,99.3


In [2]:
d1_d10_gap = (result_table.loc['D1'] - result_table.loc['D10']).to_frame().T
d1_d10_gap.index = ['D1-D10 gap (percentage points)']
display(d1_d10_gap.round(2))

Service,Electricity,Safe drinking water,Internet use
Year,2023,2023,2024
D1-D10 gap (percentage points),30.55,3.88,73.89


## How to read the table

For each decile, the percentage is the share of its population living in countries where national service coverage is below 50%. For example, the D1 electricity value is the share of the global poorest decile living in low-electrification countries. It is **not** the share of D1 individuals without electricity.

The data-coverage table shows the share of each decile's population whose country has an indicator value. The reported percentages use that observed population as the denominator.

## Blog paragraph

The paragraph below is generated from the table above so its figures stay aligned with the calculations.

In [3]:
d1 = result_table.loc['D1']
d10 = result_table.loc['D10']
electricity_year = indicator_sources['Electricity']['year']
water_year = indicator_sources['Safe drinking water']['year']
internet_year = indicator_sources['Internet use']['year']

blog_paragraph = (
    f"The table shows the share of each global income decile living in countries where fewer than half the population has access to electricity or safe drinking water, or uses the internet. In the poorest decile, these shares are {d1[('Electricity', electricity_year)]:.1f}% for electricity ({electricity_year}), {d1[('Safe drinking water', water_year)]:.1f}% for safe drinking water ({water_year}), and {d1[('Internet use', internet_year)]:.1f}% for internet use ({internet_year}). By the richest decile, the corresponding shares are {d10[('Electricity', electricity_year)]:.2f}%, {d10[('Safe drinking water', water_year)]:.2f}%, and {d10[('Internet use', internet_year)]:.2f}%. This shows that the poorest decile is disproportionately concentrated in countries with low service coverage, especially for internet use. The figures describe where people live, not which individuals lack a service, and they do not establish whether the same people experience multiple deprivations."
)
print(blog_paragraph)

The table shows the share of each global income decile living in countries where fewer than half the population has access to electricity or safe drinking water, or uses the internet. In the poorest decile, these shares are 30.6% for electricity (2023), 3.9% for safe drinking water (2023), and 74.0% for internet use (2024). By the richest decile, the corresponding shares are 0.02%, 0.02%, and 0.11%. This shows that the poorest decile is disproportionately concentrated in countries with low service coverage, especially for internet use. The figures describe where people live, not which individuals lack a service, and they do not establish whether the same people experience multiple deprivations.


## Exploratory indicator: adult literacy

The WDI adult literacy series (`WB_WDI_SE_ADT_LITR_ZS`) is based on surveys and censuses, so observations are not annual. For the 2024 D1 distribution, this section uses each country's **latest observation on or before 2024** (for example, Nigeria has 2024 data and the Democratic Republic of the Congo has 2023 data).

This improves population coverage substantially, but the observation years differ across countries. The table therefore reports the median observation year and the share of each decile covered. Coverage is 99.2% for D1, but only 37.7% for D10; interpret results for the richest deciles cautiously.

As elsewhere, the measure counts people living in countries where **national adult literacy is below 50%**. It does not identify which individuals are illiterate.

In [4]:
LITERACY_REFERENCE_YEAR = 2024
literacy_source = fetch_indicator('WB_WDI', 'WB_WDI_SE_ADT_LITR_ZS')
literacy_source = (
    literacy_source[literacy_source['year'].le(LITERACY_REFERENCE_YEAR)]
    .sort_values('year')
    .drop_duplicates('code', keep='last')
    .rename(columns={'year': 'literacy_year', 'indicator_value': 'adult_literacy_pct'})
)

literacy_distribution = distribution[distribution['year'].eq(LITERACY_REFERENCE_YEAR)].sort_values('welf', kind='stable').copy()
literacy_total_population = literacy_distribution['pop'].sum()
literacy_distribution['decile'] = np.minimum(
    np.ceil(literacy_distribution['pop'].cumsum() / (literacy_total_population / 10)),
    10,
).astype(int)
literacy_distribution = literacy_distribution.merge(
    literacy_source[['code', 'literacy_year', 'adult_literacy_pct']],
    on='code',
    how='left',
)

literacy_rows = []
for decile, decile_group in literacy_distribution.groupby('decile'):
    observed = decile_group.dropna(subset=['adult_literacy_pct'])
    total_decile_population = decile_group['pop'].sum()
    observed_population = observed['pop'].sum()
    missing_population = total_decile_population - observed_population
    below_threshold_population = observed.loc[
        observed['adult_literacy_pct'].lt(50), 'pop'
    ].sum()
    median_observation_year = observed.drop_duplicates('code')['literacy_year'].median()
    literacy_rows.append({
        'Decile': f'D{decile}',
        'Observed countries below 50% literacy (%)': 100 * below_threshold_population / observed_population,
        'Population coverage (%)': 100 * observed_population / total_decile_population,
        'Lower bound (% of full decile)': 100 * below_threshold_population / total_decile_population,
        'Upper bound (% of full decile)': 100 * (below_threshold_population + missing_population) / total_decile_population,
        'Median literacy observation year': median_observation_year,
    })
literacy_table = pd.DataFrame(literacy_rows).set_index('Decile')
print(f'Adult literacy: WDI; 2024 decile distribution matched to each country\'s latest observation on or before {LITERACY_REFERENCE_YEAR}.')
display(literacy_table.round(1))

Adult literacy: WDI; 2024 decile distribution matched to each country's latest observation on or before 2024.


,Observed countries below 50% literacy (%),Population coverage (%),Lower bound (% of full decile),Upper bound (% of full decile),Median literacy observation year
Decile,,,,,
D1,13.4,99.2,13.3,14.1,2021.0
D2,6.1,99.7,6.0,6.3,2021.0
D3,2.7,99.8,2.7,3.0,2021.0
D4,1.8,99.7,1.8,2.1,2021.0
D5,1.4,99.5,1.4,1.9,2021.0
D6,1.0,99.1,0.9,1.8,2021.0
D7,0.5,97.5,0.5,3.0,2021.0
D8,0.2,90.4,0.2,9.8,2021.0
D9,0.1,67.5,0.1,32.6,2021.0


## D1–D10 gaps across indicators

The gap is the D1 share minus the D10 share, in percentage points. For literacy, country observations are the latest available through 2024; its interval accounts for missing country data. FCS uses the FY26 list mapped to calendar 2025.

In [5]:
gap_rows = []
for service, source in indicator_sources.items():
    year = source['year']
    d1_share = result_table.loc['D1', (service, year)]
    d10_share = result_table.loc['D10', (service, year)]
    d1_coverage_service = coverage_table.loc['D1', (service, year)]
    d10_coverage_service = coverage_table.loc['D10', (service, year)]
    d1_lower = d1_share * d1_coverage_service / 100
    d1_upper = d1_lower + (100 - d1_coverage_service)
    d10_lower = d10_share * d10_coverage_service / 100
    d10_upper = d10_lower + (100 - d10_coverage_service)
    gap_rows.append({
        'Indicator': service,
        'Reference': str(year),
        'D1 share (%)': d1_share,
        'D10 share (%)': d10_share,
        'D1-D10 gap (pp)': d1_share - d10_share,
        'D1 data coverage (%)': d1_coverage_service,
        'D10 data coverage (%)': d10_coverage_service,
        'Gap lower bound (pp)': d1_lower - d10_upper,
        'Gap upper bound (pp)': d1_upper - d10_lower,
    })

literacy_d1 = literacy_table.loc['D1']
literacy_d10 = literacy_table.loc['D10']
gap_rows.append({
    'Indicator': 'Adult literacy',
    'Reference': 'Latest per country through 2024',
    'D1 share (%)': literacy_d1['Observed countries below 50% literacy (%)'],
    'D10 share (%)': literacy_d10['Observed countries below 50% literacy (%)'],
    'D1-D10 gap (pp)': literacy_d1['Observed countries below 50% literacy (%)'] - literacy_d10['Observed countries below 50% literacy (%)'],
    'D1 data coverage (%)': literacy_d1['Population coverage (%)'],
    'D10 data coverage (%)': literacy_d10['Population coverage (%)'],
    'Gap lower bound (pp)': literacy_d1['Lower bound (% of full decile)'] - literacy_d10['Upper bound (% of full decile)'],
    'Gap upper bound (pp)': literacy_d1['Upper bound (% of full decile)'] - literacy_d10['Lower bound (% of full decile)'],
})

from parse_fcs_list import parse_fcs
FCS_PDF = NOTEBOOK_DIR.parent / 'data' / 'C2-FCSList-FY06toFY26.pdf'
fcs_codes_2025 = set(parse_fcs(FCS_PDF).query('fiscal_year == 2026')['code'])
fcs_distribution = distribution[distribution['year'].eq(2025)].sort_values('welf', kind='stable').copy()
fcs_total_population = fcs_distribution['pop'].sum()
fcs_distribution['decile'] = np.minimum(
    np.ceil(fcs_distribution['pop'].cumsum() / (fcs_total_population / 10)),
    10,
).astype(int)
fcs_shares = fcs_distribution.groupby('decile').apply(
    lambda group: 100 * group.loc[group['code'].isin(fcs_codes_2025), 'pop'].sum() / group['pop'].sum(),
    include_groups=False,
)
gap_rows.append({
    'Indicator': 'Fragile and conflict-affected countries',
    'Reference': '2025 (FY26 FCS list)',
    'D1 share (%)': fcs_shares.loc[1],
    'D10 share (%)': fcs_shares.loc[10],
    'D1-D10 gap (pp)': fcs_shares.loc[1] - fcs_shares.loc[10],
    'D1 data coverage (%)': 100.0,
    'D10 data coverage (%)': 100.0,
    'Gap lower bound (pp)': fcs_shares.loc[1] - fcs_shares.loc[10],
    'Gap upper bound (pp)': fcs_shares.loc[1] - fcs_shares.loc[10],
})

gap_summary = pd.DataFrame(gap_rows).set_index('Indicator')
display(gap_summary.round(2))

,Reference,D1 share (%),D10 share (%),D1-D10 gap (pp),D1 data coverage (%),D10 data coverage (%),Gap lower bound (pp),Gap upper bound (pp)
Indicator,,,,,,,,
Electricity,2023,30.57,0.02,30.55,99.98,98.78,29.33,30.56
Safe drinking water,2023,3.90,0.02,3.88,99.98,97.79,1.67,3.90
Internet use,2024,74.00,0.11,73.89,90.58,99.74,66.66,76.33
Adult literacy,Latest per country through 2024,13.37,0.01,13.36,99.21,37.69,-49.04,14.06
Fragile and conflict-affected countries,2025 (FY26 FCS list),55.76,0.35,55.41,100.00,100.00,55.41,55.41


## D1 concentration relative to the world

For each indicator, this compares the share of the D1 living in low-coverage or FCS countries with the corresponding share of the observed world population. The ratio is `D1 share / world share`; 5x means the D1 is five times as likely as the average person to live in those countries. Literacy uses its latest country observation through 2024 and is especially sensitive to missing data.

In [6]:
ratio_rows = []
for service, source in indicator_sources.items():
    year = source['year']
    bins = distribution[distribution['year'].eq(year)].sort_values('welf', kind='stable').copy()
    bins['decile'] = np.minimum(
        np.ceil(bins['pop'].cumsum() / (bins['pop'].sum() / 10)),
        10,
    ).astype(int)
    values = fetch_indicator(source['dataset'], source['indicator'])
    values = values[values['year'].eq(year)].drop_duplicates('code').set_index('code')['indicator_value']
    bins = bins.merge(values.rename('national_rate'), left_on='code', right_index=True, how='left')
    observed = bins.dropna(subset=['national_rate'])
    world_coverage = 100 * observed['pop'].sum() / bins['pop'].sum()
    world_share = 100 * observed.loc[observed['national_rate'].lt(LOW_COVERAGE_THRESHOLD), 'pop'].sum() / observed['pop'].sum()
    d1_share = result_table.loc['D1', (service, year)]
    d1_coverage_service = coverage_table.loc['D1', (service, year)]
    ratio_rows.append({
        'Indicator': service,
        'Reference': str(year),
        'D1 share (%)': d1_share,
        'World share (%)': world_share,
        'D1 / world ratio': d1_share / world_share,
        'D1 population covered (%)': d1_coverage_service,
        'World population covered (%)': world_coverage,
    })

literacy_world_observed = literacy_distribution.dropna(subset=['adult_literacy_pct'])
literacy_world_coverage = 100 * literacy_world_observed['pop'].sum() / literacy_distribution['pop'].sum()
literacy_world_share = 100 * literacy_world_observed.loc[
    literacy_world_observed['adult_literacy_pct'].lt(LOW_COVERAGE_THRESHOLD), 'pop'
].sum() / literacy_world_observed['pop'].sum()
literacy_d1_all = literacy_distribution[literacy_distribution['decile'].eq(1)]
literacy_d1_observed = literacy_d1_all.dropna(subset=['adult_literacy_pct'])
literacy_d1_share = 100 * literacy_d1_observed.loc[
    literacy_d1_observed['adult_literacy_pct'].lt(LOW_COVERAGE_THRESHOLD), 'pop'
].sum() / literacy_d1_observed['pop'].sum()
literacy_d1_coverage = 100 * literacy_d1_observed['pop'].sum() / literacy_d1_all['pop'].sum()
ratio_rows.append({
    'Indicator': 'Adult literacy',
    'Reference': 'Latest per country through 2024',
    'D1 share (%)': literacy_d1_share,
    'World share (%)': literacy_world_share,
    'D1 / world ratio': literacy_d1_share / literacy_world_share,
    'D1 population covered (%)': literacy_d1_coverage,
    'World population covered (%)': literacy_world_coverage,
})

fcs_codes_2025 = set(parse_fcs(FCS_PDF).query('fiscal_year == 2026')['code'])
fcs_world_share = 100 * fcs_distribution.loc[
    fcs_distribution['code'].isin(fcs_codes_2025), 'pop'
].sum() / fcs_distribution['pop'].sum()
fcs_d1_share = fcs_shares.loc[1]
ratio_rows.append({
    'Indicator': 'Fragile and conflict-affected countries',
    'Reference': '2025 (FY26 FCS list)',
    'D1 share (%)': fcs_d1_share,
    'World share (%)': fcs_world_share,
    'D1 / world ratio': fcs_d1_share / fcs_world_share,
    'D1 population covered (%)': 100.0,
    'World population covered (%)': 100.0,
})

ratio_summary = pd.DataFrame(ratio_rows).set_index('Indicator')
display(ratio_summary.round(2))

,Reference,D1 share (%),World share (%),D1 / world ratio,D1 population covered (%),World population covered (%)
Indicator,,,,,,
Electricity,2023,30.57,4.77,6.40,99.98,99.69
Safe drinking water,2023,3.90,0.84,4.65,99.98,99.51
Internet use,2024,74.00,15.02,4.93,90.58,97.12
Adult literacy,Latest per country through 2024,13.37,3.03,4.42,99.21,89.01
Fragile and conflict-affected countries,2025 (FY26 FCS list),55.76,13.49,4.13,100.00,100.00


# This could also work... how much more likely are you to live in country with low access if you are in D1 vs. world and what is that gap - turns out it usually between 4x and 6x